# 🧭 Four-Objective qLogEHVI Campaign

This notebook demonstrates the v1.1.1 generalized multi-objective workflow with four coupled objectives. Every observed row records all objective values for the same experiment, and qLogEHVI suggests candidates against the configured reference point.

The 50-row loop is intentionally compact, but four-objective qLogEHVI is more expensive than the smaller examples and may take several minutes. Treat this as a workflow demonstration, not a qLogEHVI runtime benchmark.

In [ ]:
from pathlib import Path
from shutil import copyfile
import os
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

mpl_cache = PROJECT_ROOT / ".matplotlib-cache"
mpl_cache.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from bo_forge.session import CampaignSession

CONFIG_PATH = PROJECT_ROOT / "configs/11_four_objective_mixed_constrained_qlogehvi.yaml"
SEED_LOG_PATH = PROJECT_ROOT / "examples/11_four_objective_mixed_constrained_campaign_log.csv"
WORKING_LOG_PATH = PROJECT_ROOT / "examples/11_four_objective_mixed_constrained_working_log.csv"
LATEST_SUGGESTIONS_PATH = PROJECT_ROOT / "examples/11_four_objective_mixed_constrained_latest_suggestions.csv"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORT_PATH = REPORTS_DIR / "11_four_objective_campaign_report.txt"
PARETO_PLOT_PATH = REPORTS_DIR / "11_four_objective_pairwise_pareto.pdf"
PARALLEL_PLOT_PATH = REPORTS_DIR / "11_four_objective_parallel_pareto.pdf"
HYPERVOLUME_PLOT_PATH = REPORTS_DIR / "11_four_objective_hypervolume.pdf"
TARGET_OBSERVED_ROWS = 50

copyfile(SEED_LOG_PATH, WORKING_LOG_PATH)

campaign = CampaignSession.from_files(CONFIG_PATH, WORKING_LOG_PATH)
campaign.validate()
display(campaign.summary())

## 🎯 Campaign Trade-Offs

The campaign balances two objectives to maximise (`yield`, `selectivity`) and two objectives to minimise (`waste`, `energy_use`). The synthetic simulator creates real trade-offs: high-yield regions tend to need more loading and energy, Water tends to reduce waste while changing selectivity, and high base equivalents are constrained at high catalyst loading.

In [ ]:
display(campaign.next_action())
display(campaign.pareto_summary())
display(campaign.pareto_front()[["row_id", "yield", "selectivity", "waste", "energy_use", "solvent"]])

## 🧪 Coupled Four-Objective Simulator

The simulator returns all objective values together for each proposed design. v1.1.1 still assumes coupled objective evaluation; missing or asynchronous objective values are deferred.

In [ ]:
def simulate_coupled_objectives(row: pd.Series) -> dict[str, float]:
    loading = float(row["catalyst_loading"])
    reaction_time = float(row["reaction_time"])
    base = float(row["base_equivalents"])
    solvent = str(row["solvent"])

    loading_scaled = (loading - 0.02) / 0.18
    time_scaled = (reaction_time - 20.0) / 70.0
    base_penalty = abs(base - 1.0)

    solvent_yield = {"MeCN": 0.08, "DMF": 0.03, "Water": -0.05}[solvent]
    solvent_selectivity = {"MeCN": -0.02, "DMF": 0.05, "Water": 0.12}[solvent]
    solvent_waste = {"MeCN": 0.08, "DMF": 0.14, "Water": -0.10}[solvent]
    solvent_energy = {"MeCN": 0.03, "DMF": 0.08, "Water": -0.04}[solvent]

    yield_peak = np.exp(
        -0.5
        * (
            ((loading_scaled - 0.64) / 0.18) ** 2
            + ((time_scaled - 0.68) / 0.24) ** 2
        )
    )
    selectivity_peak = np.exp(
        -0.5
        * (
            ((loading_scaled - 0.42) / 0.22) ** 2
            + ((time_scaled - 0.48) / 0.25) ** 2
        )
    )

    values = {
        "yield": 0.28 + 0.55 * yield_peak - 0.07 * base_penalty + solvent_yield,
        "selectivity": (
            0.30
            + 0.46 * selectivity_peak
            + 0.07 * base_penalty
            + solvent_selectivity
            - 0.08 * yield_peak
        ),
        "waste": 0.34 + 0.38 * loading_scaled + 0.12 * base_penalty + solvent_waste - 0.10 * time_scaled,
        "energy_use": 0.20 + 0.52 * time_scaled + 0.18 * loading_scaled + 0.08 * base_penalty + solvent_energy,
    }
    return {name: float(np.clip(value, 0.02, 0.98)) for name, value in values.items()}

## 🔁 Run A Compact BO Loop

The loop keeps the usual BO Forge rhythm: suggest, append, simulate the coupled experiment, mark observed, reload. It stops at 50 observed rows including the seed data.

In [ ]:
run_records = []

while len(campaign.observed_data()) < TARGET_OBSERVED_ROWS:
    remaining = TARGET_OBSERVED_ROWS - len(campaign.observed_data())
    suggestions = campaign.suggest_next(batch_size=min(2, remaining))
    suggestions.to_csv(LATEST_SUGGESTIONS_PATH, index=False)
    campaign.append_suggestions(suggestions)

    for _, suggestion in suggestions.iterrows():
        objective_values = simulate_coupled_objectives(suggestion)
        campaign.mark_observed(
            row_id=str(suggestion["row_id"]),
            objective_values=objective_values,
        )
        run_records.append(
            {
                "row_id": suggestion["row_id"],
                "solvent": suggestion["solvent"],
                **objective_values,
            }
        )
    campaign.reload()

run_log = pd.DataFrame(run_records)
display(run_log.tail())
print(f"Observed rows: {len(campaign.observed_data())}")

## 📊 Pareto State After 50 Rows

The Pareto front is computed in the full four-objective space. Pairwise plots are projections of that same full-space front, not separate 2D Pareto fronts.

In [ ]:
display(campaign.summary())
display(campaign.pareto_summary())
pareto = campaign.pareto_front()
display(pareto[["row_id", "yield", "selectivity", "waste", "energy_use", "solvent"]].head(12))

In [ ]:
campaign.plot_hypervolume(save_path=HYPERVOLUME_PLOT_PATH)

In [ ]:
campaign.plot_pareto(save_path=PARETO_PLOT_PATH)

In [ ]:
campaign.plot_pareto_parallel(save_path=PARALLEL_PLOT_PATH)

## 📝 Report Export

In [ ]:
report = campaign.report()
campaign.export_report(REPORT_PATH)
report.keys()

## 💻 Compact CLI Demo

The CLI can inspect the same campaign state. This is intentionally a short inspection demo; the 50-row campaign loop above uses the Python session API to keep the notebook fast and readable.

In [ ]:
cli_common = [
    sys.executable,
    "-m",
    "bo_forge",
    "pareto-summary",
    "--config",
    str(CONFIG_PATH),
    "--log",
    str(WORKING_LOG_PATH),
]
completed = subprocess.run(
    cli_common,
    cwd=PROJECT_ROOT,
    check=True,
    text=True,
    capture_output=True,
)
print(completed.stdout)